# Tiền Xử Lý Video Front View cho Nhận Diện Ký Hiệu

Quy trình chuẩn hóa dữ liệu video đầu vào qua 2 giai đoạn chính:
1. **Temporal Boundary Localization (TBL):** Định vị thời gian, loại bỏ các khung hình tĩnh đầu/cuối và chỉ giữ lại phân đoạn thực hiện ký hiệu.
2. **Spatial Crop & Resize:** Cắt tập trung khu vực Đầu - Vai - Eo để giảm nhiễu nền và nén về kích thước chuẩn `224x224`.


⚙️ 1. Khởi Tạo Thư Viện & Cấu Hình MediaPipe
Import các thư viện xử lý ảnh/video (`OpenCV`, `MediaPipe`) và định nghĩa các điểm mốc (landmarks) xương khớp cần thiết cho thuật toán TBL.


In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import os
import math

mp_pose = mp.solutions.pose

# Danh sách 6 điểm mốc liên quan đến tay/vai 
LANDMARKS = {
    "LShoulder": mp_pose.PoseLandmark.LEFT_SHOULDER,
    "RShoulder": mp_pose.PoseLandmark.RIGHT_SHOULDER,
    "LElbow": mp_pose.PoseLandmark.LEFT_ELBOW,
    "RElbow": mp_pose.PoseLandmark.RIGHT_ELBOW,
    "LWrist": mp_pose.PoseLandmark.LEFT_WRIST,
    "RWrist": mp_pose.PoseLandmark.RIGHT_WRIST,
}

## 2. Thuật Toán Định Vị Thời Gian (TBL)
Các hàm bổ trợ tính toán góc khuỷu tay và xác định trạng thái hoạt động (active/inactive) của người dùng dựa trên vị trí các khớp vai và cổ tay.


In [ ]:
def _angle_2d(a, b, c):
    """Tính góc ABC (độ) trong mặt phẳng 2D, luôn trả về int."""
    ba_x, ba_y = a[0] - b[0], a[1] - b[1]
    bc_x, bc_y = c[0] - b[0], c[1] - b[1]

    norm_ba = math.hypot(ba_x, ba_y)
    norm_bc = math.hypot(bc_x, bc_y)
    if norm_ba == 0 or norm_bc == 0:
        return 0

    cos_val = (ba_x * bc_x + ba_y * bc_y) / (norm_ba * norm_bc)
    cos_val = max(-1.0, min(1.0, cos_val))
    return int(round(math.degrees(math.acos(cos_val))))

def frame_active_from_landmarks(lm, theta=160, vis_th=0.6, point_th=0.5):
    """TBL: Kiểm tra frame active bằng cách tính góc khuỷu tay."""
    # Trả về 0 (inactive) nếu bất kỳ điểm nào ở tay/vai bị mờ/khuất
    if any(float(lm[idx].visibility) < vis_th for idx in LANDMARKS.values()):
        return 0

    elbow_pairs = [
        (mp_pose.PoseLandmark.LEFT_SHOULDER, mp_pose.PoseLandmark.LEFT_ELBOW, mp_pose.PoseLandmark.LEFT_WRIST),
        (mp_pose.PoseLandmark.RIGHT_SHOULDER, mp_pose.PoseLandmark.RIGHT_ELBOW, mp_pose.PoseLandmark.RIGHT_WRIST),
    ]

    angles = []
    for s_idx, e_idx, w_idx in elbow_pairs:
        s_lm, e_lm, w_lm = lm[s_idx], lm[e_idx], lm[w_idx]
        if min(s_lm.visibility, e_lm.visibility, w_lm.visibility) < point_th:
            continue
        angle = _angle_2d((s_lm.x, s_lm.y), (e_lm.x, e_lm.y), (w_lm.x, w_lm.y))
        if angle > 0:
            angles.append(angle)

    if not angles:
        return 0
    
    # Lấy góc trung bình của cả 2 tay
    elbow_angle = int(round(sum(angles) / len(angles)))
    return 1 if elbow_angle < theta else 0

## 3. Hàm Tiền Xử Lý Video Đơn Lẻ (Crop & Resize)
Tích hợp quy trình 2 bước:
* **Pass 1:** Quét luồng video để xác định các phân đoạn chuyển động hợp lệ (tối thiểu 0.67 giây).
* **Pass 2:** Tìm Bounding Box chuẩn quanh cơ thể tại khung hình giữa (mid-frame), áp dụng padding 0.4s và cắt/nén video vuông 1:1 (`224x224`).


In [ ]:
def process_single_front_video(video_id, src_video_path, output_root, theta=160, target_size=224):
    """
    Quy trình xử lý luồng (Stream) cho 1 video Front duy nhất:
    Pass 1: Tìm ranh giới thời gian active.
    Pass 2: Cắt không gian vùng đầu/vai cố định, nén về 224x224 và xuất file.
    """
    try:
        # ===== PASS 1: ĐỌC LUỒNG ĐỂ XÁC ĐỊNH RANH GIỚI THỜI GIAN =====
        cap = cv2.VideoCapture(src_video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0: fps = 25.0
            
        s_raw = []
        with mp_pose.Pose(static_image_mode=False, model_complexity=1) as pose:
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret: break
                
                # Chuyển màu trực tiếp và xử lý qua MediaPipe
                result = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                if not result.pose_landmarks:
                    s_raw.append(0)
                    continue
                    
                # Gọi hàm logic kiểm tra góc khuỷu tay
                lm = result.pose_landmarks.landmark
                is_active = frame_active_from_landmarks(lm, theta=theta)
                s_raw.append(is_active)
        cap.release()
        
        # Trích xuất các đoạn hành động liên tục từ chuỗi nhị phân s_raw
        segments = []
        i, n = 0, len(s_raw)
        while i < n:
            if s_raw[i] == 0:
                i += 1
                continue
            j = i
            while j < n and s_raw[j] == 1: j += 1
            segments.append((i, j - 1))
            i = j
            
        # ==================== ĐOẠN SỬA ĐỔI: GỘP CÁC ĐOẠN QUÁ GẦN NHAU ====================
        # Nếu khoảng cách giữa đoạn trước và đoạn sau < 0.8 giây (nhiễu nhấp nháy), gộp lại làm một.
        max_gap_frames = int(round(fps * 0.8)) 
        merged_segments = []
        
        if segments:
            current_start, current_end = segments[0]
            for next_start, next_end in segments[1:]:
                # Kiểm tra khoảng cách giữa frame kết thúc đoạn trước và frame bắt đầu đoạn sau
                if next_start - current_end <= max_gap_frames:
                    # Tiến hành nối dài đoạn hiện tại
                    current_end = next_end
                else:
                    # Nếu khoảng cách quá xa, lưu đoạn cũ lại và chuyển sang đoạn mới
                    merged_segments.append((current_start, current_end))
                    current_start, current_end = next_start, next_end
            merged_segments.append((current_start, current_end))
        else:
            merged_segments = []
        # ==============================================================================

        # Lọc các đoạn đạt chuẩn thời lượng tối thiểu t_min = 0.67 giây 
        # (Thay "segments" bằng "merged_segments" mới được gộp)
        valid_segments = []
        for start_idx, end_idx in merged_segments:
            duration = ((end_idx + 1) - start_idx) / fps
            if duration >= 0.67:
                valid_segments.append((start_idx, end_idx))
                
        if len(valid_segments) == 0:
            return f"⏩ Skip ID {video_id}: Không tìm thấy đoạn active hợp lệ."
            
        elif len(valid_segments) > 1:
            # Video bị đứt gãy thành nhiều phần -> Coi như dữ liệu lỗi, loại bỏ luôn!
            return f"⏩ Skip ID {video_id}: Phát hiện {len(valid_segments)} phân đoạn (nghi ngờ lỗi/nhiễu) -> Đã loại bỏ."
        

        # ===== PASS 2: CẮT KHÔNG GIAN (CROP) VÀ RESIZE CHO TỪNG ĐOẠN ACTIVE =====
        # Tạo thư mục đầu ra lưu video
        os.makedirs(output_root, exist_ok=True)
        
        for seg_idx, (start_frame, end_frame) in enumerate(valid_segments):
            cap = cv2.VideoCapture(src_video_path)
            w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) 
            
            # 1. Tính khung hình ở giữa dựa trên đoạn active gốc TRƯỚC để Bounding Box chuẩn nhất
            mid_frame_idx = (start_frame + end_frame) // 2
            
            # Tính toán số lượng frame padding (Ví dụ chọn 0.4s cho cả trước và sau)
            padding_frames = int(round(fps * 0.4))
            
            # CẬP NHẬT: Cộng thêm padding vào TRƯỚC thời điểm bắt đầu (Chống tràn dưới 0)
            start_frame = max(0, start_frame - padding_frames)
            
            # Cộng thêm padding vào SAU phân đoạn chuyển động (Chống tràn quá tổng số frame)
            if total_frames > 0:
                end_frame = min(total_frames - 1, end_frame + padding_frames)
            else:
                end_frame = end_frame + padding_frames
            
            cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame_idx)
            ret, mid_frame = cap.read()
            
            # Thiết lập tọa độ mặc định phòng khi MediaPipe lỗi
            box_size = min(h, w)
            x1, y1 = (w - box_size) // 2, (h - box_size) // 2
            
            if ret:
                with mp_pose.Pose(static_image_mode=True, model_complexity=1) as pose_detector:
                    res = pose_detector.process(cv2.cvtColor(mid_frame, cv2.COLOR_BGR2RGB))
                    if res.pose_landmarks:
                        lm = res.pose_landmarks.landmark
                        pixel_nose_x = int(lm[mp_pose.PoseLandmark.NOSE].x * w)
                        pixel_l_sh_x = int(lm[mp_pose.PoseLandmark.LEFT_SHOULDER].x * w)
                        pixel_r_sh_x = int(lm[mp_pose.PoseLandmark.RIGHT_SHOULDER].x * w)
                        pixel_sh_y_avg = int(((lm[mp_pose.PoseLandmark.LEFT_SHOULDER].y + lm[mp_pose.PoseLandmark.RIGHT_SHOULDER].y) / 2) * h)
                        
                        shoulder_width = abs(pixel_l_sh_x - pixel_r_sh_x)
                        pixel_nose_y = int(lm[mp_pose.PoseLandmark.NOSE].y * h)
                        
                        # Tăng hệ số lên 3.6 để khung hình bự hơn nữa, bao phủ hết eo và cả một phần đùi
                        box_size = int(shoulder_width * 3.6) 
                        # Vẫn phải đảm bảo không vượt quá kích thước video gốc
                        box_size = min(box_size, min(h, w))
                        
                        x1 = pixel_nose_x - box_size // 2
                        # Đẩy y1 lên cao sát đầu (cách mũi tầm ngắn hơn 0.6) để phần dưới dôi ra cực nhiều,
                        # như vậy không gian thả tay xuống qua eo và ngang hông sẽ được giữ lại hết
                        y1 = pixel_nose_y - int(shoulder_width * 0.6)
            
            # Thuật toán chống tràn viền giữ nguyên tỉ lệ vuông 1:1
            if y1 < 0: y1 = 0
            if y1 + box_size > h: y1 = h - box_size
            if x1 < 0: x1 = 0
            if x1 + box_size > w: x1 = w - box_size
            
            # Khởi tạo bộ ghi video đầu ra chuẩn 224x224
            out_path = os.path.join(output_root, f"{video_id}_{seg_idx:02d}.mp4")
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out_writer = cv2.VideoWriter(out_path, fourcc, fps, (target_size, target_size))
            
            # Tiến hành tua đến vị trí start_frame và cắt (vòng lặp range này tự động chạy đến end_frame mới đã nới rộng)
            cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
            for f_idx in range(start_frame, end_frame + 1):
                ret_f, frame = cap.read()
                if not ret_f: break
                
                cropped = frame[y1:y1+box_size, x1:x1+box_size]
                resized = cv2.resize(cropped, (target_size, target_size), interpolation=cv2.INTER_AREA)
                out_writer.write(resized)
                
            cap.release()
            out_writer.release()
            
        return f"✅ Đã xử lý thành công ID: {video_id}"
    except Exception as e:
        return f"❌ Lỗi tại ID {video_id}: {str(e)}"

## 4. Thực Thi Tiền Xử Lý Hàng Loạt (Batch Processing)
Quét toàn bộ thư mục gốc `VSL_FULL_FRONT`, tự động áp dụng quy trình tiền xử lý tuần tự cho từng video và xuất kết quả vào thư mục `VSL_FULL_FRONT_CROPPED_TO224x224_V2`.


In [ ]:
import glob
import os
import concurrent.futures
from tqdm import tqdm

# IMPORT HÀM WORKER TỪ FILE UTILS VỪA TẠO
from worker_utils import video_worker

if __name__ == "__main__":
    input_root = "..\\VSL_FULL_FRONT_RAW"
    all_videos = glob.glob(os.path.join(input_root, "*", "*.mp4"))
    
    num_workers = os.cpu_count() # Tự động lấy tối đa số nhân CPU của máy bạn
    
    print(f"🚀 Khởi động xử lý ĐA NHÂN (Parallel Processing) trên Windows...")
    print(f"🔥 Số lượng nhân CPU sử dụng: {num_workers}")
    print(f"📦 Tổng số video cần xử lý: {len(all_videos)}")
    print("-" * 50)
    
    thong_ke = {"thanh_cong": 0, "bo_qua_qua_ngan": 0, "loi_he_thong": 0}
    
    # Kích hoạt đa nhân xử lý song song thông qua ProcessPoolExecutor
    with concurrent.futures.ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(video_worker, path): path for path in all_videos}
        
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Đang xử lý"):
            try:
                res = future.result()
                if "✅" in res:
                    thong_ke["thanh_cong"] += 1
                elif "⏩ Skip" in res:
                    thong_ke["bo_qua_qua_ngan"] += 1
                else:
                    thong_ke["loi_he_thong"] += 1
            except Exception:
                thong_ke["loi_he_thong"] += 1
                
    print("\n" + "="*15 + " BÁO CÁO KẾT QUẢ CROP VIDEO (ĐA NHÂN) " + "="*15)
    print(f" Tổng số video quét:    {len(all_videos)} video")
    print(f" Đã crop thành công:     {thong_ke['thanh_cong']} video")
    print(f" Bị bỏ qua (quá ngắn):   {thong_ke['bo_qua_qua_ngan']} video")
    print(f" Bị lỗi hệ thống:        {thong_ke['loi_he_thong']} video")
    print("=" * 60)